# Week 2: Advanced Pandas Data Wrangling
## Notebook 11: Point-in-Time asof Joins & Financial Resampling

### 1. Conceptual Intuition & Theoretical Background
In financial trading and real-time fraud detection, joining two streaming datasets by exact timestamp match fails because quotes, order book updates, and trades occur asynchronously at microsecond intervals.

**1. Point-in-Time As-Of Joins (`pd.merge_asof`):**
Matches records where the right timestamp is the **most recent prior** timestamp:
$$\text{Timestamp}_{\text{quote}} \le \text{Timestamp}_{\text{trade}}$$
- Prevents **Lookahead Data Leakage** by guaranteeing that ML features only reflect information available strictly prior to trade execution.

**2. Financial Candlestick Resampling:**
Converts continuous tick streams into fixed time intervals (e.g. 5-minute candles) using Open, High, Low, Close (OHLC) aggregation.

---

### 2. Scratch Python Implementation: As-Of Join & OHLC Resampler

In [ ]:
import pandas as pd
import numpy as np

# Construct Simulated Quotes Stream (Bid/Ask updates)
quote_times = pd.date_range("2026-10-30 09:30:00", periods=8, freq="750ms")
quotes_df = pd.DataFrame({
    'timestamp': quote_times,
    'bid': [150.10, 150.12, 150.11, 150.15, 150.14, 150.18, 150.20, 150.19],
    'ask': [150.15, 150.17, 150.16, 150.20, 150.19, 150.22, 150.25, 150.24]
})

# Construct Simulated Trade Execution Stream (Asynchronous arrival)
trade_times = pd.to_datetime([
    "2026-10-30 09:30:01.100",
    "2026-10-30 09:30:02.400",
    "2026-10-30 09:30:04.800"
])
trades_df = pd.DataFrame({
    'timestamp': trade_times,
    'trade_price': [150.14, 150.18, 150.22],
    'volume': [100, 250, 50]
})

# 1. Execute Point-in-Time As-Of Join (Direction: 'backward')
# Matches trade with prevailing Bid/Ask strictly before trade execution
enriched_trades = pd.merge_asof(
    trades_df,
    quotes_df,
    on='timestamp',
    direction='backward'
)

# 2. Resample Tick Stream into OHLC Bars
tick_stream = pd.DataFrame({
    'price': np.random.normal(150.0, 0.5, 60).round(2)
}, index=pd.date_range("2026-10-30 09:30:00", periods=60, freq="10s"))

ohlc_bars = tick_stream['price'].resample('30s').ohlc()

print("--- Enriched Trades via Point-in-Time As-Of Join ---")
print(enriched_trades[['timestamp', 'trade_price', 'bid', 'ask', 'volume']])
print("\n--- Resampled 30-Second OHLC Bars ---")
print(ohlc_bars.head(4))

### 3. Step-by-Step Code Walkthrough
1. **As-Of Matching:** Matches each trade with the most recent prior quote row.
2. **Lookahead Leakage Prevention:** `direction='backward'` guarantees no future quotes are matched.
3. **`.resample('30s').ohlc()`:** Aggregates price streams into standard financial candles.

---

### 4. Industry Applications & Trade-offs
- **Algorithmic Execution & Slippage Analysis:** Measuring price execution relative to prevailing National Best Bid and Offer (NBBO).
- **Trade-off:** Both DataFrames must be strictly sorted by the join key before executing `merge_asof`.

---

### 5. Top Interview Questions & Answers
**Q1: What is the risk of using standard `pd.merge` with rounded timestamps instead of `merge_asof`?**
> **Answer:** Rounding timestamps can align trades with quotes that occurred in the future (lookahead bias), leading to overly optimistic backtest results that fail in live production.

**Q2: What is the `tolerance` argument in `pd.merge_asof`?**
> **Answer:** It defines the maximum allowable time difference for a valid match. If no quote exists within the tolerance window, NaN is returned.